# Reproducción del detector EEG de dos escalas

Código científico original de Breves CV, revisión pilot_cudnn_train_mode_1.
El resultado histórico fue 114/168 crisis. Una nueva ejecución no garantiza repetirlo exactamente.
Primero lee README.md y METODO.md del repositorio. No representa validación clínica.
Cinco modelos con 12 pacientes para entrenar, cuatro para validar y cuatro para probar en cada ronda.
Se conservan 590 EDF y 168 crisis de 20 personas. Los cinco modelos se fijan antes de probar.
Este cuaderno descarga y verifica el paquete original. No cambia etiquetas, particiones o hiperparámetros.


## 1. Conectar Drive y fijar las carpetas de este ensayo


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, importlib.metadata, importlib.util, json, os, shutil, subprocess, sys, zipfile
drive.mount('/content/drive')
ROOT = Path('/content/tesis_breves_cv_corregido')
BACKUP = Path('/content/drive/MyDrive/detector_eeg')
DRIVE_DATASET = Path('/content/drive/MyDrive/base de datos')
DATASET_DIR = Path('/content/cache_pediatrica_ssd')
ROOT.mkdir(exist_ok=True); BACKUP.mkdir(parents=True, exist_ok=True)
RAM_CACHE_GIB = 24
print('Originales:', DRIVE_DATASET)
print('Avance y resultados:', BACKUP)


## 2. Comprobar dependencias y GPU

Se conserva PyTorch del entorno de Colab. Solo se instala lo que falte.
Al reanudar se comparan las versiones y la GPU con el respaldo.


In [ ]:
modules = {'numpy':'numpy', 'scipy':'scipy', 'scikit-learn':'sklearn',
           'psutil':'psutil', 'matplotlib':'matplotlib'}
missing = [name for name,module in modules.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
import torch, numpy as np, scipy, sklearn, psutil, matplotlib
if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU en Entorno de ejecución antes de continuar.')
env_record = {'versions': {'python': f'{sys.version_info.major}.{sys.version_info.minor}',
              **{name: importlib.metadata.version(name) for name in
                 ('torch','numpy','scipy','scikit-learn','psutil','matplotlib')}},
              'gpu': torch.cuda.get_device_name(0)}
env_path = BACKUP/'entorno.json'
if env_path.exists() and json.loads(env_path.read_text()) != env_record:
    print('Entorno guardado:', env_path.read_text())
    print('Entorno actual:', env_record)
    raise RuntimeError('El entorno cambió. Conserva Drive y recupera las versiones/GPU guardadas antes de reanudar.')
print('GPU:', env_record['gpu'])
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,1), 'GiB')
print('RAM total/libre:', round(psutil.virtual_memory().total/1024**3,1), '/',
      round(psutil.virtual_memory().available/1024**3,1), 'GiB')
print(env_record['versions'])


## 3. Descargar el código original y comprobar su huella

La descarga tiene menos de 0.2 MB. No incluye señales ni artículos.


In [ ]:
PACKAGE_NAME = 'Paquete_Colab_Breves_CV_Corregido.zip'
PACKAGE_SHA256 = '0c5e937b18bfcd8c5359d3dffec3b555173f08fdf0d6c884817437dc00d1f63c'
package_path = BACKUP/PACKAGE_NAME
if not package_path.exists():
    import urllib.request
    url = 'https://raw.githubusercontent.com/Armandogma/detector-eeg/main/codigo_congelado.zip'
    raw = urllib.request.urlopen(url, timeout=60).read()
    if hashlib.sha256(raw).hexdigest() != PACKAGE_SHA256:
        raise ValueError('La huella del código no coincide. No se ejecutará.')
    package_path.write_bytes(raw)
    del raw
if hashlib.sha256(package_path.read_bytes()).hexdigest() != PACKAGE_SHA256:
    raise ValueError('ZIP de otra versión. No se reemplazará el respaldo existente.')
with zipfile.ZipFile(package_path) as archive:
    if archive.testzip() or len(archive.namelist()) != len(set(archive.namelist())):
        raise ValueError('ZIP dañado o con rutas duplicadas.')
    meta = json.loads(archive.read('paquete_manifest.json'))
    if meta['family'] != 'tcn_gru_breves_cv_v1' or meta.get('software_revision') != 'pilot_cudnn_train_mode_1':
        raise ValueError('Paquete de otro experimento.')
    if set(archive.namelist()) != set(meta['files']) | {'paquete_manifest.json'}:
        raise ValueError('Inventario del ZIP distinto.')
    old = ROOT/'paquete_manifest.json'
    if old.exists() and old.read_bytes() != archive.read('paquete_manifest.json'):
        raise ValueError('Esta carpeta temporal pertenece a otra versión. Conserva los archivos.')
    for item in archive.infolist():
        name = item.filename
        if '\\' in name or ':' in name or not (ROOT/name).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError('Ruta no permitida en el ZIP.')
        if (item.external_attr>>16)&0o170000 == 0o120000:
            raise ValueError('El ZIP contiene enlaces no permitidos.')
        raw = archive.read(name)
        if name in meta['files'] and hashlib.sha256(raw).hexdigest() != meta['files'][name]:
            raise ValueError('Contenido distinto: '+name)
    archive.extractall(ROOT)
if not env_path.exists(): env_path.write_text(json.dumps(env_record, indent=2))
print('Paquete verificado. Señales preparadas previstas, GB:',
      {k:round(v/1e9,2) for k,v in meta['planned_cache_bytes'].items()})
print((ROOT/'LEEME.md').read_text()[:2100])


## 4. Verificar la base y recuperar solo archivos ausentes

Primero comprueba la copia de EDF del SSD. Si está completa y coincide con el manifiesto,
la reutiliza. Si falta algo, busca la base en `Mi unidad/base de datos` y copia solo los
archivos requeridos. No borra archivos originales ni carpetas extra.


In [ ]:
# Usa ROOT, DATASET_DIR y DRIVE_DATASET definidos en la primera celda.
from pathlib import Path
import hashlib, json, shutil, subprocess

cohort = json.loads((ROOT/'analisis/cohorte_pediatrica.json').read_text())
manifest = json.loads((ROOT/'analisis/manifiesto_dataset.json').read_text())
eligible = set(cohort['eligible_paths'])
records = [r for r in manifest['records'] if r['path'] in eligible]
if len(records) != 590 or sum(len(r['events']) for r in records) != 168:
    raise ValueError('Inventario pediátrico distinto. No se cambiará la población.')

# Para reutilizar el SSD se comprueban tamaño y cabecera de cada EDF,
# y el contenido íntegro de las anotaciones y resúmenes. No se releen 45 GB.
requirements = {}
for record in records:
    requirements[record['path']] = dict(size=record['file_bytes'],
        header_bytes=record['header']['hbytes'], sha256=record['header_sha256'])
    requirements[record['case']+'/'+record['case']+'-summary.txt'] = dict(sha256=record['summary_sha256'])
    if record.get('annotation_sha256'):
        requirements[record['path']+'.seizures'] = dict(sha256=record['annotation_sha256'])
if any('..' in Path(name).parts or Path(name).is_absolute() for name in requirements):
    raise ValueError('Ruta no permitida en el inventario.')

def files_to_recover(folder):
    invalid = []
    for name, expected in requirements.items():
        path = folder/name
        try:
            if not path.is_file() or ('size' in expected and path.stat().st_size != expected['size']):
                invalid.append(name)
                continue
            with path.open('rb') as stream:
                raw = stream.read(expected['header_bytes']) if 'header_bytes' in expected else stream.read()
            if hashlib.sha256(raw).hexdigest() != expected['sha256']:
                invalid.append(name)
        except OSError:
            invalid.append(name)
    return invalid

DATASET_DIR = Path(DATASET_DIR)
pending = files_to_recover(DATASET_DIR)
if not pending:
    print('La copia SSD ya contiene los 590 EDF elegibles y sus anotaciones verificadas.')
    print('Se reutiliza sin volver a copiar la base desde Drive.')
else:
    print('Archivos ausentes o incompatibles que se deben recuperar:', len(pending))
    source = Path(DRIVE_DATASET)
    probe = 'chb01/chb01_01.edf'
    if not (source/probe).is_file():
        # Búsqueda de carpetas hasta tres niveles dentro de Mi unidad.
        # No se recorren los EDF, ni se elige por resultados del modelo.
        mydrive = Path('/content/drive/MyDrive')
        candidates = set()
        if (mydrive/probe).is_file():
            candidates.add(mydrive.resolve())
        for pattern in ('*/chb01', '*/*/chb01', '*/*/*/chb01'):
            for folder in mydrive.glob(pattern):
                if (folder/'chb01_01.edf').is_file():
                    candidates.add(folder.parent.resolve())
            if candidates:
                break
        if len(candidates) != 1:
            print('Ruta configurada:', source)
            print('Carpetas encontradas:', *sorted(map(str, candidates)), sep='\n')
            raise FileNotFoundError(
                'No se encontró una única base en Drive. Abre su carpeta en el panel Archivos, '
                'copia la ruta y asigna DRIVE_DATASET = Path(r"ruta_copiada"). '
                'Después vuelve a ejecutar esta celda. No es necesario subir EDF desde la laptop.')
        source = next(iter(candidates))
        print('Carpeta de la base localizada:', source)
    DRIVE_DATASET = source
    if source.resolve() == DATASET_DIR.resolve():
        raise ValueError('La copia SSD está incompleta. Indica la carpeta ORIGINAL de Drive como DRIVE_DATASET.')
    absent_at_source = [name for name in pending if not (source/name).is_file()]
    if absent_at_source:
        raise FileNotFoundError('También faltan archivos en esta carpeta de Drive: '+str(absent_at_source[:5]))
    if shutil.which('rsync') is None:
        subprocess.run(['apt-get', '-qq', 'update'], check=True)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'rsync'], check=True)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    copy_list = ROOT/'archivos_elegibles.txt'
    copy_list.write_text('\n'.join(sorted(pending))+'\n')
    subprocess.run(['rsync', '-a', '--ignore-times', '--info=progress2', '--partial',
                    '--files-from='+str(copy_list), str(source)+'/', str(DATASET_DIR)+'/'], check=True)
    pending = files_to_recover(DATASET_DIR)
    if pending:
        raise ValueError('Archivos aún ausentes o distintos del inventario: '+str(pending[:5]))
    print('Copia completada y verificada. Originales conservados.')
print('Base que utilizará el modelo:', DATASET_DIR)
print('Población: 20 personas, 590 EDF y 168 crisis. Cinco rondas por persona, cada una 12/4/4.')
print('Disco libre:', round(shutil.disk_usage(ROOT).free/1024**3, 1), 'GiB')

# Solo reutiliza características sin etiquetas ni pesos de la sesión anterior.
from pathlib import Path


def reusar_caracteristicas_breves(root, previous_root):
    root, previous_root = Path(root), Path(previous_root)
    cache, previous = root/'preparados', previous_root/'preparados'
    if cache.exists():
        print('Se conserva la caché de esta revisión:', cache)
        return False
    if not previous.is_dir():
        print('No está la caché de la sesión anterior. prepare calculará las características.')
        return False
    import json
    cohort = json.loads((root/'analisis/cohorte_pediatrica.json').read_text(encoding='utf-8'))
    paths = cohort['eligible_paths']
    if len(paths) != 590:
        raise ValueError('Inventario distinto; no reutilizar esta caché.')
    for name in paths:
        path = previous/(name.replace('/', '__')+'.breves.npy')
        if not path.is_file() or not path.with_suffix('.json').is_file():
            print('La caché anterior está incompleta. Se preparará una caché separada.')
            return False
    # El programa prepare verificará contratos y SHA-256 antes de usar cada matriz.
    # No se reutilizan protocolos, pilotos, modelos o resultados del intento anterior.
    cache.symlink_to(previous.resolve(), target_is_directory=True)
    print('Características anteriores disponibles sin duplicarlas:', cache)
    print('prepare verificará sus huellas. Los resultados y fuentes anteriores se conservan.')
    return True

reusar_caracteristicas_breves(ROOT, Path('/content/tesis_breves_cv'))


## 5. Preparar las características de 2 s y 4 s

Se calculan una sola vez sobre los 590 EDF. Son transformaciones causales de cada registro
y su pasado, sin etiquetas ni ajuste conjunto entre personas. Compartir esas características
entre rondas no comparte pesos de redes o etiquetas de prueba.
La caché nueva ocupa unos 9 GiB. El límite RAM es 24 GiB con reserva 8 GiB para trabajar en
Colab con unos 51 GB. No se requiere llenar toda la RAM.


In [ ]:
import json
import subprocess
import sys
from collections import deque
from datetime import datetime, timezone


def mostrar_pilotos():
    for path in sorted((ROOT/'entrenamiento').glob('grupo*/piloto.json')):
        try:
            value = json.loads(path.read_text(encoding='utf-8'))
            print(path.parent.name, {
                name: value.get(name) for name in
                ('passed', 'initial_loss', 'final_loss', 'weights_discarded')
            }, flush=True)
        except (OSError, ValueError) as error:
            print('No se pudo leer', path, str(error), flush=True)


def run_stage(stage):
    if stage not in ('prepare', 'pilot', 'train', 'evaluate', 'verify'):
        raise ValueError('Etapa no reconocida.')
    command = [sys.executable, '-u', str(ROOT/'codigo/colab_breves_cv.py'),
               '--stage', stage, '--drive', str(BACKUP), '--dataset', str(DATASET_DIR),
               '--ram-gib', str(RAM_CACHE_GIB)]
    # Añade cada intento al registro. Conserva los anteriores y las fuentes congeladas.
    log_path = BACKUP/('salida_'+stage+'.log')
    BACKUP.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=30)
    print('Salida completa en:', log_path, flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- '+stage+' '+datetime.now(timezone.utc).isoformat()+' ---\n')
        log.flush()
        with subprocess.Popen(command, cwd=str(ROOT), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                code = process.wait()
            except BaseException:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                log.write('\nCelda interrumpida; proceso detenido.\n')
                log.flush()
                raise
        log.write('\nCódigo de salida: '+str(code)+'\n')
    if code:
        mostrar_pilotos()
        raise RuntimeError('Se detuvo '+stage+' (código '+str(code)+'). Salida guardada en '
                           +str(log_path)+'. Últimas líneas:\n'+''.join(tail))

run_stage('prepare')


## 6. Comprobar aprendizaje de un lote TRAIN por ronda

Cada piloto usa solamente TRAIN de esa ronda. Debe bajar la pérdida al menos 20% en 60
pasos. Sus pesos se descartan. El piloto no mide detección en pacientes desconocidos.
Un fallo queda guardado y detiene la corrida.
La salida completa se muestra en la celda y queda en Drive como `salida_pilot.log`.
Si falla, conserva ese archivo y no borres un `piloto.json` rechazado para volver a intentarlo.


In [ ]:
run_stage('pilot')


## 7. Entrenar las cinco rondas y fijar los cinco modelos

Se entrenan 15 épocas completas por ronda con progreso visible por lotes y validación.
Todas las probabilidades de validación por época quedan guardadas para poder revisar la
selección después. El avance se respalda por época completa. Todavía no se calculan
predicciones de prueba. No modifiques fuentes o configuración durante la corrida.


In [ ]:
run_stage('train')


## 8. Evaluar las cinco pruebas externas

La celda comprueba que los cinco modelos y sus reglas estén congelados antes de probar.
Reportará todas las rondas, aunque alguna falle los objetivos de validación. Cada paciente
se prueba con su modelo reservado y el resultado conjunto será detectadas / 168.
La celda es reanudable y reutiliza las probabilidades de prueba ya guardadas.

La auditoría profunda opcional `run_stage('verify')` reproduce las selecciones de todas
las épocas desde validación. Puede tardar. No busca umbrales con prueba.


In [ ]:
run_stage('evaluate')


## 9. Consultar y conservar resultados

El diagnóstico también se guarda en Drive aunque falle la descarga del navegador.
`avance.zip` incluye pesos y recuperación; `diagnostico.zip` incluye tablas, figuras y
probabilidades de todas las épocas sin pesos. Conserva ambos. Si la sesión se reinicia,
abre este mismo cuaderno desde el principio y recupera el avance de la misma familia.


In [ ]:
from IPython.display import display, Image
output = ROOT/'entrenamiento'
state_path = output/'estado_cv.json'
if state_path.exists():
    print('ESTADO:', json.loads(state_path.read_text()))
for fold in range(1, 6):
    folder = output/('grupo'+str(fold))
    selected = folder/'seleccion.json'
    if selected.exists():
        s = json.loads(selected.read_text()); t = s['total']
        print(f"Ronda {fold}: VALIDACIÓN {t['detected']}/{t['events']} | {t['false_alarms_per_hour']:.3f} FA/h | cumple objetivos: {s['validation_goals_met']} | época {s['best_epoch']}")
result_path = output/'evaluacion_cv.json'
if result_path.exists():
    result = json.loads(result_path.read_text()); t = result['total']
    print(f"\nPRUEBA CRUZADA: {t['detected']}/{t['events']} crisis | perdidas {t['events']-t['detected']}")
    print(f"Sensibilidad {100*t['sensitivity']:.2f}% | falsas alarmas/h {t['false_alarms_per_hour']:.3f}")
    print(f"Precisión por evento {100*t['event_precision']:.2f}% | F1 {t['event_f1']:.4f}")
    print(f"Tiempo bajo alarma {100*t['time_under_alarm']:.3f}% | cobertura {100*t['coverage']:.3f}%")
    print('Demora mediana solo entre detectadas:', t['delay_median'], 's')
    print('Inicio dentro de crisis:', result['onset_only_total']['detected'], '/', t['events'])
    for person, value in result['by_person'].items():
        print(person, f"{value['detected']}/{value['events']} | {value['false_alarms_per_hour']:.3f} FA/h")
    print('Resultados por duración:', result['event_duration_groups'])
else:
    print('Aún no existe un resultado externo completo; se conserva el diagnóstico parcial.')
for name in ('aprendizaje', 'validacion', 'personas', 'grupos', 'demoras', 'duraciones', 'alarmas'):
    path = output/(name+'.png')
    if path.exists(): display(Image(filename=str(path)))
diagnostic = BACKUP/'diagnostico.zip'
with zipfile.ZipFile(diagnostic, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in sorted(output.rglob('*')):
        if path.is_file() and '.partial' not in path.name and path.suffix in ('.json', '.csv', '.png', '.npz'):
            archive.write(path, 'entrenamiento/'+path.relative_to(output).as_posix())
    for path in sorted(BACKUP.glob('salida_*.log')):
        archive.write(path, 'salidas/'+path.name)
print('Respaldo con pesos:', BACKUP/'avance.zip')
print('Diagnóstico sin pesos:', diagnostic)
print('Evaluación exploratoria. CV no borra la exposición histórica a prueba.')
try:
    files.download(str(diagnostic))
except Exception as error:
    print('El archivo quedó guardado en Drive aunque falló la descarga:', error)
